In [0]:
from pyspark.sql.functions import date_trunc, count, countDistinct

SILVER_TABLE = "gharchive_dev.v1_pyspark.gharchive_silver"
GOLD_TABLE = "gharchive_dev.v1_pyspark.gharchive_gold_activity_counts"

# Read from silver
df_silver = spark.read.table(SILVER_TABLE)

# Aggregate: hourly event counts by event type
df_gold = (
    df_silver
    .withColumn("event_hour", date_trunc("hour", "created_at"))
    .groupBy("event_hour", "type")
    .agg(
        count("*").alias("event_count"),
        countDistinct("actor_login").alias("unique_actors"),
        countDistinct("repo_name").alias("unique_repos")
    )
    .orderBy("event_hour", "type")
)

# Write as Delta table
df_gold.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)

print(f"Gold table written: {GOLD_TABLE}")
print(f"Row count: {spark.read.table(GOLD_TABLE).count()}")

In [0]:
from pyspark.sql.functions import count, countDistinct, col

SILVER_TABLE = "gharchive_dev.v1_pyspark.gharchive_silver"
GOLD_TABLE_REPOS = "gharchive_dev.v1_pyspark.gharchive_gold_top_repos"

# NOTE: Reads the ENTIRE silver table again — full scan, full recompute
df_silver = spark.read.table(SILVER_TABLE)

df_top_repos = (
    df_silver
    .groupBy("repo_name")
    .agg(
        count("*").alias("total_events"),
        countDistinct("actor_login").alias("unique_contributors"),
        countDistinct("type").alias("event_types")
    )
    .orderBy(col("total_events").desc())
)

df_top_repos.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE_REPOS)

print(f"Gold table written: {GOLD_TABLE_REPOS}")
print(f"Row count: {spark.read.table(GOLD_TABLE_REPOS).count()}")

In [0]:
from pyspark.sql.functions import count, countDistinct, min, max, col

SILVER_TABLE = "gharchive_dev.v1_pyspark.gharchive_silver"
GOLD_TABLE_ACTORS = "gharchive_dev.v1_pyspark.gharchive_gold_top_actors"

# NOTE: Reads the ENTIRE silver table AGAIN — another full scan
df_silver = spark.read.table(SILVER_TABLE)

df_top_actors = (
    df_silver
    .groupBy("actor_login")
    .agg(
        count("*").alias("total_events"),
        countDistinct("repo_name").alias("unique_repos"),
        countDistinct("type").alias("event_types"),
        min("created_at").alias("first_event"),
        max("created_at").alias("last_event")
    )
    .orderBy(col("total_events").desc())
)

df_top_actors.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE_ACTORS)

print(f"Gold table written: {GOLD_TABLE_ACTORS}")
print(f"Row count: {spark.read.table(GOLD_TABLE_ACTORS).count()}")